In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.feature_selection import SelectFromModel 
from sklearn.ensemble import RandomForestClassifier 

from sklearn.metrics import mean_absolute_error

In [ ]:
train_and_target = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

train = train_and_target.drop('SalePrice', axis = 1)
target = train_and_target['SalePrice']

object_cols = list(train.select_dtypes('object').columns)

categorical_cols = ['MSSubClass','OverallQual','OverallCond','YearBuilt','YearRemodAdd',
                      'BsmtFullBath','BsmtHalfBath','FullBath','HalfBath','BedroomAbvGr','KitchenAbvGr',
                      'TotRmsAbvGrd','Fireplaces','GarageYrBlt','GarageCars','MoSold','YrSold']

numerical_cols = [n for n in list(train.columns) if n not in object_cols + categorical_cols and n != 'Id']

In [ ]:
to_drop=['PoolQC','MiscFeature','Alley','Fence','FireplaceQu']
train=train.drop(to_drop,axis=1)
test=test.drop(to_drop,axis=1)

object_cols = [n for n in object_cols if n not in to_drop]
categorical_cols = [n for n in categorical_cols if n not in to_drop]
numerical_cols = [n for n in numerical_cols if n not in to_drop]

In [ ]:
#define object pipeline
obj_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse=False))
])

#define categorical, non-object pipeline
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent'))
])

# define numerical pipeline
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', MinMaxScaler())
])

# combine object, cat, and numerical pipelines
preprocessor = ColumnTransformer([
    ('obj', obj_pipe, object_cols),
    ('cat', cat_pipe, categorical_cols),
    ('num', num_pipe, numerical_cols)
])

preprocessor.fit(train)

obj_columns = preprocessor.named_transformers_['obj']['encoder'].get_feature_names(object_cols)
new_columns = np.append(obj_columns, categorical_cols)
new_columns = np.append(new_columns, numerical_cols)

new_train = pd.DataFrame(preprocessor.transform(train), columns = new_columns)
new_test=pd.DataFrame(preprocessor.transform(test), columns = new_columns)

In [ ]:
select = SelectFromModel(
        RandomForestClassifier(n_estimators=100, random_state=42),
        threshold="4*median")

select.fit(new_train, target)
train_l1 = select.transform(new_train)
test_l1=select.transform(new_test)

In [ ]:
my_model = XGBRegressor(n_estimators=500,
                           learning_rate=.1,
                           early_stopping_rounds=5,
                             max_depth=3).fit(train_l1, target)

predicted_prices = my_model.predict(test_l1)

my_submission = pd.DataFrame({'Id': test.Id, 'SalePrice': predicted_prices})
my_submission.to_csv('submission.csv', index=False)

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(train_l1,target)

In [ ]:
y_pred=my_model.predict(X_test)
mean_absolute_error(y_test,y_pred)